# DAY 156 - Hyperparameter Tuning.
@A.IPYNB

# The Art of Tuning: Beyond Guesswork

### The Problem
A Deep Learning model is like a safe with 10 combination dials.
* **Learning Rate:** Too high (explode), too low (slow).
* **Batch Size:** Too big (OOM), too small (noisy).
* **Layers/Neurons:** Too many (overfit), too few (underfit).

Trying every combination (**Grid Search**) is impossible ($10^6$ experiments).
Trying random combinations (**Random Search**) is inefficient.

### The Solution: Bayesian Optimization (TPE)
We use **Optuna**.
Optuna uses a **Tree-structured Parzen Estimator (TPE)**.
1.  It treats the hyperparameter search as a probability problem.
2.  It looks at past trials.
3.  It builds a probability model of *which values are likely to lower the loss*.
4.  It intelligently picks the next set of parameters to try.



**The Result:** It finds the "State of the Art" configuration in 10% of the time.

In [1]:
# @title 1. Optuna
# Optuna is the SOTA library for hyperparameter optimization
!pip install -q optuna

import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Tuning on: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 16.7 MB/s eta 0:00:00
Tuning on: cuda


### Defining the Search Space
Instead of hard-coding values, we define distributions.

* **Learning Rate:** Logarithmic scale ($1e^{-5}$ to $1e^{-1}$).
* **Optimizer:** Categorical choice (Adam, SGD, RMSprop).
* **Layers:** Integer choice (1 to 3 layers).
* **Neurons:** Integer choice (32 to 128).

In [3]:
def get_data():
    # Load FashionMNIST (Harder than MNIST)
    transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
    train_data = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
    test_data = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)

    # Subsample for speed during tuning (we don't need all 60k images to know if a LR is bad)
    subset_indices = np.random.choice(len(train_data), 5000, replace=False)
    train_subset = Subset(train_data, subset_indices)

    return train_subset, test_data

def objective(trial):
    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)
    # Here we pick an optimizer
    optimizer_name = trial.suggest_categorical("optimizer", ["Adam", "SGD"])
    n_units = trial.suggest_int("n_units", 32, 128)
    dropout_rate = trial.suggest_float("dropout", 0.1, 0.5)

    # Model Building
    model = nn.Sequential(
        nn.Flatten(),
        nn.Linear(28*28, n_units),
        nn.ReLU(),
        nn.Dropout(dropout_rate),
        nn.Linear(n_units, 10)
    ).to(device)

    # Here we get optimizer based on suggestion
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr)

    criterion = nn.CrossEntropyLoss()
    train_data, test_data = get_data()
    train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
    test_loader = DataLoader(test_data, batch_size=64)

    model.train()
    for epoch in range(5): # Only training for 5 epochs to test viability
        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            output = model(data)
            loss = criterion(output, target)
            loss.backward()
            optimizer.step()

        # Pruning (This is Optional but Powerful):
        # If the model is trash at epoch 2, we kill it immediately.
        # This saves massive compute.
        model.eval()
        correct = 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                pred = model(data).argmax(dim=1, keepdim=True)
                correct += pred.eq(target.view_as(pred)).sum().item()

        accuracy = correct / len(test_loader.dataset)

        # THIS Reports intermediate result to Optuna
        trial.report(accuracy, epoch)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return accuracy # We want to MAXIMIZE this

In [4]:
# @title 3. Optimization
# We create a "Study" and ask Optuna to maximize accuracy.

study = optuna.create_study(direction="maximize")

# n_trials=20 means we will train 20 different models
study.optimize(objective, n_trials=20, timeout=600) # Max 10 minutes

print("\n--- Tuning Complete ---")
print(f"Best Accuracy: {study.best_value:.4f}")
print("Best Params:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

[I 2026-02-04 13:37:48,976] A new study created in memory with name: no-name-2307f769-b662-40ae-9b90-d7ade2c05b61
100%|██████████| 26.4M/26.4M [00:00<00:00, 93.5MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 3.65MB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 61.2MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 13.6MB/s]
[I 2026-02-04 13:38:07,656] Trial 0 finished with value: 0.4742 and parameters: {'lr': 0.0005619050874955229, 'optimizer': 'SGD', 'n_units': 74, 'dropout': 0.12713924166099175}. Best is trial 0 with value: 0.4742.
[I 2026-02-04 13:38:23,239] Trial 1 finished with value: 0.7415 and parameters: {'lr': 0.012836843714887497, 'optimizer': 'SGD', 'n_units': 116, 'dropout': 0.2894616571689244}. Best is trial 1 with value: 0.7415.
[I 2026-02-04 13:38:39,156] Trial 2 finished with value: 0.6717 and parameters: {'lr': 0.045351508530306006, 'optimizer': 'SGD', 'n_units': 41, 'dropout': 0.4964213185100491}. Best is trial 1 with value: 0.7415.
[I 2026-02-04 13:38:55,454] Trial 3


--- Tuning Complete ---
Best Accuracy: 0.8161
Best Params:
  lr: 0.004213684421403308
  optimizer: Adam
  n_units: 90
  dropout: 0.22005414402464923


### Visualization: What Matters?
One of the most powerful features of Optuna is identifying **Parameter Importance**.
Does Learning Rate matter more than Batch Size? Optuna will tell you.

In [5]:
# @title 4. Visualization
from optuna.visualization import plot_optimization_history, plot_param_importances

# 1. Optimization History
# Shows if the model improved over trials
plot_optimization_history(study).show()

# 2. Parameter Importance
# Shows which hyperparameter had the biggest impact on accuracy.
plot_param_importances(study).show()

### Analysis of Results
Looking at the generated plots, we gained massive insights that a grid search would have missed:

1.  **Optimization History:** The red line shows the "Best Value" climbed rapidly in the first few trials and plateaued around **82% accuracy**. This tells us that further tuning on *these* specific parameters has diminishing returns—we hit the ceiling for this architecture.
2.  **Parameter Importance:** This is the game-changer.
    * **Optimizer (43%):** The choice between `Adam` vs `SGD` was the single most critical factor.
    * **Dropout (35%):** Regularization was the second most important lever.
    * **Learning Rate (14%) & Neurons (8%):** Surprisingly, the number of neurons mattered very little.

### The Strategic Win
If you were tuning this manually, you might have spent days tweaking the `learning_rate` or adding more `n_units`, thinking that was the key.
**Optuna proved that was a waste of time.** The real gains were in the **Optimizer choice**.

By using Bayesian Optimization, you:
1.  **Saved Time:** Stopped bad trials early (Pruning).
2.  **Gained Knowledge:** Learned *what* actually matters (Importance).
3.  **Maximized Performance:** Found the statistical ceiling of your model.

You are no longer guessing; you are engineering.